# Ablation Study — Statistical Summary & Inference

This notebook takes the raw cross-validation results from the ablation experiment and produces:

- aggregated summary statistics (mean/std + confidence intervals)
- performance deltas vs the baseline (full feature set)
- statistical significance tests (paired tests across folds with multiple-comparison correction)
- a feature impact ranking (importance by performance drop)

**Input:** `ablation_results.csv`  
**Outputs:**
- `ablation_summary_stats.csv`
- `ablation_significance_tests.csv`
- `ablation_feature_importance.csv`

## 1) Load results and validate format

We expect one row per **fold × model variant** with the following columns:

- identifiers: `model_variant`, `features_removed`, `fold`
- metrics: `r2`, `adjusted_r2`, `rmse`, `mae`, `mse`, `mape`

The baseline variant is named **`Baseline`** and should appear once per fold.

In [11]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

INPUT_CSV = "ablation_results.csv"

OUT_STATS = "ablation_summary_stats.csv"
OUT_SIG = "ablation_significance_tests.csv"
OUT_IMP = "ablation_feature_importance.csv"

METRICS = ["r2", "adjusted_r2", "rmse", "mae", "mse", "mape"]
BASELINE_NAME = "Baseline"

CONF_LEVEL = 0.95
ALPHA = 0.05
CORRECTION = "holm"     
PRIMARY_METRIC = "mae" 


In [12]:
results_df = pd.read_csv(INPUT_CSV)

# Normalize 'features_removed' (baseline uses 'none')
results_df["features_removed"] = (
    results_df["features_removed"]
      .fillna("none")
      .astype(str)
      .str.strip()
)
# Treat common null-like strings as 'none'
results_df.loc[
    results_df["features_removed"].str.lower().isin(["none", "nan", ""]),
    "features_removed"
] = "none"
print("Shape:", results_df.shape)
print("Columns:", results_df.columns.tolist())
print("# folds:", results_df["fold"].nunique(), "| folds:", sorted(results_df["fold"].unique()))
print("Variants:", results_df["model_variant"].unique().tolist())

baseline_rows = (results_df["model_variant"] == BASELINE_NAME).sum()
print("Baseline rows:", baseline_rows)

coverage = results_df.groupby("model_variant")["fold"].nunique()
print("Fold coverage (min/max):", int(coverage.min()), int(coverage.max()))
results_df.head()


Shape: (25, 11)
Columns: ['model_variant', 'features_removed', 'fold', 'r2', 'adjusted_r2', 'rmse', 'mae', 'mse', 'mape', 'delta_r2', 'delta_rmse']
# folds: 5 | folds: [1, 2, 3, 4, 5]
Variants: ['Baseline', '- engine_power', '- mileage', '- age', '- brand_encoded']
Baseline rows: 5
Fold coverage (min/max): 5 5


,model_variant,features_removed,fold,r2,adjusted_r2,rmse,mae,mse,mape,delta_r2,delta_rmse
0,Baseline,none,1,0.662425,0.623845,2160.239640,1669.407526,4.666635e+06,8.655815,NaN,NaN
1,- engine_power,engine_power,1,0.608928,0.576338,2325.118700,1891.436489,5.406177e+06,9.868951,-0.053497,164.879060
2,- mileage,mileage,1,0.638999,0.608915,2233.937951,1777.622177,4.990479e+06,9.255545,-0.023426,73.698312
3,- age,age,1,0.003544,-0.079494,3711.467963,2909.134103,1.377499e+07,15.029315,-0.658881,1551.228324
4,- brand_encoded,brand_encoded,1,0.667625,0.639927,2143.536505,1662.009304,4.594749e+06,8.598064,0.005200,-16.703135


## Summary statistics and confidence intervals
We aggregate each metric across CV folds for each ablation variant and compute a t-based confidence interval for the mean.

In [13]:
def mean_ci_t(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan, np.nan)
    mu = x.mean()
    se = stats.sem(x)
    tcrit = stats.t.ppf(1 - (1-confidence)/2, df=n-1)
    return (mu - tcrit * se, mu + tcrit * se)

rows = []
for (variant, feat_removed), g in results_df.groupby(["model_variant", "features_removed"]):
    row = {"model_variant": variant, "features_removed": feat_removed, "n_folds": g["fold"].nunique()}
    for m in METRICS:
        vals = g[m].to_numpy()
        row[f"{m}_mean"] = float(np.mean(vals))
        row[f"{m}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan
        ci_lo, ci_hi = mean_ci_t(vals, confidence=CONF_LEVEL)
        row[f"{m}_ci_lower"] = ci_lo
        row[f"{m}_ci_upper"] = ci_hi
        row[f"{m}_sem"] = float(stats.sem(vals)) if len(vals) > 1 else np.nan
        row[f"{m}_min"] = float(np.min(vals))
        row[f"{m}_max"] = float(np.max(vals))
    rows.append(row)

stats_df = pd.DataFrame(rows)

In [14]:
stats_df[["model_variant", "features_removed", "r2_mean", "rmse_mean", "mae_mean"]].sort_values("model_variant")

,model_variant,features_removed,r2_mean,rmse_mean,mae_mean
0,- age,age,0.033547,3837.144657,3244.118407
1,- brand_encoded,brand_encoded,0.713124,2085.853506,1653.046247
2,- engine_power,engine_power,0.642713,2334.115685,1900.018901
3,- mileage,mileage,0.700655,2123.553543,1693.636551
4,Baseline,none,0.709854,2097.205500,1662.729673


In [15]:
baseline_rows = stats_df[(stats_df["model_variant"] == BASELINE_NAME) & (stats_df["features_removed"] == "none")]
if len(baseline_rows) == 0:
    # Fallback if upstream used a different sentinel
    baseline_rows = stats_df[stats_df["model_variant"] == BASELINE_NAME]

if len(baseline_rows) != 1:
    raise ValueError(f"Expected exactly 1 baseline row, found {len(baseline_rows)}")

baseline_row = baseline_rows.iloc[0]

for m in METRICS:
    base = baseline_row[f"{m}_mean"]
    stats_df[f"delta_{m}"] = stats_df[f"{m}_mean"] - base
    stats_df[f"delta_{m}_pct"] = np.where(base != 0, 100 * stats_df[f"delta_{m}"] / base, np.nan)

stats_df[["model_variant", "features_removed", "delta_r2", "delta_rmse", "delta_mae"]].sort_values(
    "delta_mae", ascending=False
)


,model_variant,features_removed,delta_r2,delta_rmse,delta_mae
0,- age,age,-0.676307,1739.939157,1581.388734
2,- engine_power,engine_power,-0.067141,236.910186,237.289228
3,- mileage,mileage,-0.009199,26.348044,30.906878
4,Baseline,none,0.000000,0.000000,0.000000
1,- brand_encoded,brand_encoded,0.003270,-11.351994,-9.683426


## Significance tests (paired vs baseline)
Paired tests across folds comparing each ablation to the baseline, with Holm correction.

In [16]:
def adjust_pvalues(pvals, method="holm"):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)

    if method == "none":
        return pvals
    if method == "bonferroni":
        return np.clip(pvals * m, 0, 1)

    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = np.empty_like(ranked)

    for i, p in enumerate(ranked):
        adj[i] = (m - i) * p

    adj = np.maximum.accumulate(adj)
    adj = np.clip(adj, 0, 1)

    out = np.empty_like(adj)
    out[order] = adj
    return out


all_rows = []

for metric in METRICS:
    piv = results_df.pivot_table(index="fold", columns="model_variant", values=metric, aggfunc="mean")
    base = piv[BASELINE_NAME]
    variants = [v for v in piv.columns if v != BASELINE_NAME]

    rows = []
    pvals = []

    for v in variants:
        common = pd.concat([base, piv[v]], axis=1).dropna()
        diffs = common[v] - common[BASELINE_NAME]
        t_stat, p_val = stats.ttest_rel(common[BASELINE_NAME], common[v])

        rows.append({
            "metric": metric,
            "model_variant": v,
            "mean_diff_variant_minus_baseline": float(diffs.mean()),
            "n_pairs": int(len(common)),
            "t_stat": float(t_stat),
            "p_value": float(p_val),
        })
        pvals.append(p_val)

    tmp = pd.DataFrame(rows)
    tmp["p_value_adj"] = adjust_pvalues(tmp["p_value"].values, method=CORRECTION)
    tmp["significant"] = tmp["p_value_adj"] < ALPHA
    all_rows.append(tmp)

significance_df = pd.concat(all_rows, ignore_index=True).sort_values(["metric", "p_value_adj"])
significance_df


,metric,model_variant,mean_diff_variant_minus_baseline,n_pairs,t_stat,p_value,p_value_adj,significant
4,adjusted_r2,- age,-7.236851e-01,5,15.455744,0.000102,0.000409,True
5,adjusted_r2,- brand_encoded,1.252271e-02,5,-6.355662,0.003141,0.006699,True
6,adjusted_r2,- engine_power,-6.375560e-02,5,6.965683,0.002233,0.006699,True
7,adjusted_r2,- mileage,-9.844546e-04,5,0.120431,0.909949,0.909949,False
12,mae,- age,1.581389e+03,5,-13.294091,0.000185,0.000740,True
14,mae,- engine_power,2.372892e+02,5,-4.716464,0.009196,0.027587,True
13,mae,- brand_encoded,-9.683426e+00,5,2.478449,0.068325,0.136649,False
15,mae,- mileage,3.090688e+01,5,-1.089554,0.337153,0.337153,False
20,mape,- age,9.035576e+00,5,-12.554697,0.000232,0.000926,True
22,mape,- engine_power,1.310682e+00,5,-4.846737,0.008359,0.025077,True


## Feature importance (based on MAE)
Rank features by MAE increase when removed, and attach adjusted p-values.

In [17]:
importance_df = stats_df[stats_df["model_variant"] != BASELINE_NAME].copy()
importance_df["impact_score"] = importance_df["delta_mae"]

sig_mae = significance_df[significance_df["metric"] == PRIMARY_METRIC][
    ["model_variant", "p_value_adj", "significant"]
]

importance_df = importance_df.merge(sig_mae, on="model_variant", how="left")

importance_df = importance_df.sort_values("impact_score", ascending=False).reset_index(drop=True)
importance_df["rank"] = np.arange(1, len(importance_df) + 1)

importance_df[["rank", "features_removed", "impact_score", "p_value_adj", "significant"]]


,rank,features_removed,impact_score,p_value_adj,significant
0,1,age,1581.388734,0.000740,True
1,2,engine_power,237.289228,0.027587,True
2,3,mileage,30.906878,0.337153,False
3,4,brand_encoded,-9.683426,0.136649,False


In [18]:
stats_df.to_csv(OUT_STATS, index=False)
significance_df.to_csv(OUT_SIG, index=False)

importance_export = importance_df[["rank", "features_removed", "impact_score", "p_value_adj", "significant"]]
importance_export.to_csv(OUT_IMP, index=False)

print("Saved:", OUT_STATS, stats_df.shape)
print("Saved:", OUT_SIG, significance_df.shape)
print("Saved:", OUT_IMP, importance_export.shape)


Saved: ablation_summary_stats.csv (5, 57)
Saved: ablation_significance_tests.csv (24, 8)
Saved: ablation_feature_importance.csv (4, 5)


## 5) Deploy-check: re-train with ablation-suggested feature sets (sanity check)

This section turns the ablation ranking into a few candidate feature sets (e.g., drop the least impactful 1/3/5 features),
then re-trains + cross-validates a simple baseline pipeline to verify whether performance stays the same (or improves).

> This is **not** the final benchmark stage — it’s a quick validation that the ablation recommendation is sensible.

> Note: This works only if the ablation feature names (in `ablation_feature_importance.csv`) match the feature columns used for training here. If they don't, the notebook prints a warning and `drop_k` will be limited.


In [19]:
# Configuration
TRAIN_CSV = "data/train.csv"
TEST_CSV = "data/test.csv"
TARGET_COL = "price"
ID_COL = "carID"

CV_FOLDS = 5
SEED = 42

# Features to always exclude from modeling (identifiers, leakage fields, etc.)
DROP_ALWAYS = [ID_COL]

# How aggressively to drop "least important" features for the deploy-check:
DROP_K_LIST = [1, 3, 5]

# If True, fit the best variant on full train and export a Kaggle-style submission for test.csv
MAKE_SUBMISSION = False
SUBMISSION_OUT = "submission_ablation_best.csv"

# -----------------------
# Load train/test
# -----------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' in {TRAIN_CSV}")

X_full = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL]

# Remove always-drop columns if present
drop_existing = [c for c in DROP_ALWAYS if c in X_full.columns]
X_full = X_full.drop(columns=drop_existing)


# -----------------------
# Read ablation feature importance (from this notebook's output)
# -----------------------
imp_path = OUT_IMP  # produced above
imp = pd.read_csv(imp_path)

# Quick alignment check (ablation feature names must match training columns)
ablation_feats = imp["features_removed"].astype(str).str.strip().tolist()
matching_feats = [f for f in ablation_feats if f in X_full.columns]
if len(matching_feats) < len(ablation_feats):
    print("WARNING: Some ablation features do not match the training columns.")
    print("Ablation features:", ablation_feats)
    print("Training columns:", list(X_full.columns))
    print("Matching features:", matching_feats)

# Keep only features that actually exist in X_full
imp = imp[imp["features_removed"].isin(X_full.columns)].copy()

if imp.empty:
    raise ValueError("No ablation features matched the training columns. "
                     "Re-run ablation on the same feature columns used here.")

# Least impactful = smallest impact_score (delta MAE)
least = imp.sort_values("impact_score", ascending=True)["features_removed"].tolist()

# "Safe drop": not significant and small (or negative) MAE impact
# Threshold is relative to baseline MAE (0.1% by default).
baseline_mae = float(baseline_row["mae_mean"])
eps = 0.001 * baseline_mae
safe_drop = imp[(imp["p_value_adj"] >= ALPHA) & (imp["impact_score"] <= eps)]["features_removed"].tolist()

drop_sets = {"baseline": []}

max_k = max(DROP_K_LIST) if len(DROP_K_LIST) else 0
if len(least) < max_k:
    print(f"WARNING: Only {len(least)} ablation features matched the training columns. "
          f"drop_k variants will drop fewer than requested (max requested = {max_k}).")

for k in DROP_K_LIST:
    drop_sets[f"drop_{k}"] = least[:min(k, len(least))]

drop_sets["drop_safe"] = safe_drop

# -----------------------
# Model builder (simple but robust)
# -----------------------
def build_default_pipeline(feature_cols, df_for_dtypes):
    cat_cols = [c for c in feature_cols if df_for_dtypes[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop"
    )

    model = Ridge(alpha=1.0)

    return Pipeline(steps=[("preprocess", pre), ("model", model)])

# -----------------------
# CV evaluation
# -----------------------
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

rows = []
for name, drops in drop_sets.items():
    use_cols = [c for c in X_full.columns if c not in drops]

    pipe = build_default_pipeline(use_cols, train_df)
    scores = cross_val_score(
        pipe,
        X_full[use_cols],
        y,
        cv=kf,
        scoring="neg_mean_absolute_error"
    )
    mae = -scores
    rows.append({
        "variant": name,
        "n_features": len(use_cols),
        "dropped_features": ",".join(drops),
        "mae_mean": float(np.mean(mae)),
        "mae_std": float(np.std(mae, ddof=1)) if len(mae) > 1 else np.nan,
    })

deploy_check = pd.DataFrame(rows).sort_values("mae_mean", ascending=True)
deploy_check.to_csv("ablation_deploy_check.csv", index=False)
deploy_check


Ablation features: ['age', 'engine_power', 'mileage', 'brand_encoded']
Training columns: ['Brand', 'model', 'year', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
Matching features: ['mileage']


,variant,n_features,dropped_features,mae_mean,mae_std
0,baseline,12,,2544.600520,16.254429
4,drop_safe,12,,2544.600520,16.254429
1,drop_1,11,mileage,2702.560549,13.791862
2,drop_3,11,mileage,2702.560549,13.791862
3,drop_5,11,mileage,2702.560549,13.791862


### Optional: export a submission for the best ablation-based feature set

If `MAKE_SUBMISSION = True`, this will fit the best variant (lowest CV MAE from the table above) on the full training set,
predict `price` for `test.csv`, and save `submission_ablation_best.csv`.


In [20]:
if MAKE_SUBMISSION:
    best = deploy_check.iloc[0]
    best_variant = best["variant"]
    best_drops = drop_sets[best_variant]
    best_cols = [c for c in X_full.columns if c not in best_drops]

    pipe = build_default_pipeline(best_cols, train_df)
    pipe.fit(X_full[best_cols], y)

    preds = pipe.predict(test_df[best_cols])

    # Round to sensible integers since price is integer in the dataset
    sub = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: np.round(preds).astype(int)
    })

    sub.to_csv(SUBMISSION_OUT, index=False)
    print("Saved:", SUBMISSION_OUT, "| best variant:", best_variant)
    sub.head()


In [21]:
pd.read_csv(OUT_SIG).shape
pd.read_csv(OUT_SIG).head()


,metric,model_variant,mean_diff_variant_minus_baseline,n_pairs,t_stat,p_value,p_value_adj,significant
0,adjusted_r2,- age,-0.723685,5,15.455744,0.000102,0.000409,True
1,adjusted_r2,- brand_encoded,0.012523,5,-6.355662,0.003141,0.006699,True
2,adjusted_r2,- engine_power,-0.063756,5,6.965683,0.002233,0.006699,True
3,adjusted_r2,- mileage,-0.000984,5,0.120431,0.909949,0.909949,False
4,mae,- age,1581.388734,5,-13.294091,0.000185,0.000740,True
